# 04 — Label noise diagnostics (cleanlab)

Side analysis; **does not modify** `data/processed/`. Ranks labels that are likely wrong via
Confident Learning.

A probe (multilingual sentence embeddings + OvR LogReg on **raw** text) is trained with 5-fold
cross-validation, so every row gets an out-of-fold probability from a model that never saw it.
`cleanlab` then compares those probabilities with the given labels.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sentence_transformers import SentenceTransformer
from cleanlab.multilabel_classification.filter import find_label_issues
from cleanlab.multilabel_classification import get_label_quality_scores

import sys; sys.path.insert(0, "../experiments")  # thesis_lib żyje w experiments/
from thesis_lib import optimal_thresholds, bootstrap_f1_ci
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
TARGET_LABELS = ["radość", "smutek", "zaufanie", "wstręt", "strach", "gniew", "przeczuwanie", "zdziwienie"]
PROCESSED_DIR = Path("../data/processed")

# Dense multilingual sentence embeddings as the probe features (covers Polish).
# Loaded once; encodes RAW text (not lemmatized) so negation/context survive — a far
# better-calibrated probe than TF-IDF (validated: higher coverage, fewer false flags).
EMB_MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"
emb_model = SentenceTransformer(EMB_MODEL_NAME)
print(f"Embedding model: {EMB_MODEL_NAME}")

# Diagnose the train split of each dataset (representative, largest). text_col = RAW text.
DATASETS = {
    "TwitterEmo":    ("twitteremo_train.csv", "tekst"),
    "GoEmotions PL": ("go_emotions_train.csv", "text_pl"),
    "CLARIN-Emo":    ("clarin_emo_train.csv", "tekst"),
}

frames = {}
for name, (fname, text_col) in DATASETS.items():
    df = pd.read_csv(PROCESSED_DIR / fname)
    frames[name] = (df, text_col)
    print(f"{name:>14}: {len(df):>6} rows")

In [2]:
def compute_oof_pred_probs(X, Y, n_splits=5, random_state=RANDOM_STATE):
    """Out-of-fold predicted probabilities from a LogReg probe on dense embeddings.

    Each row is scored by a model trained on the other folds only, so the prediction is
    not contaminated by having memorized that row's (possibly wrong) label. Embeddings are
    fixed features (not fit on labels), so there is no leakage in computing them once.

    No `class_weight='balanced'`: Confident Learning relies on *calibrated* probabilities
    and already handles class imbalance via per-class confident thresholds.
    """
    n, k = Y.shape
    oof = np.zeros((n, k))
    mskf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for fold, (tr, te) in enumerate(mskf.split(X, Y), 1):
        clf = OneVsRestClassifier(LogisticRegression(max_iter=1000))
        clf.fit(X[tr], Y[tr])
        oof[te] = clf.predict_proba(X[te])
        print(f"    fold {fold}/{n_splits} done")
    return oof


def diagnose_labels(name, df, text_col, label_cols=TARGET_LABELS):
    """Encode raw text, run Confident Learning; return flags, quality scores, OOF probs."""
    texts = df[text_col].fillna("").tolist()
    Y = df[label_cols].to_numpy().astype(int)
    print(f"\n=== {name}: encoding {len(df)} rows + 5-fold OOF probe ===")
    X = emb_model.encode(texts, batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    pred_probs = compute_oof_pred_probs(X, Y)
    labels = [list(np.where(row)[0]) for row in Y]            # multilabel format: list of class-index lists
    issues = find_label_issues(labels=labels, pred_probs=pred_probs,
                               return_indices_ranked_by="self_confidence")
    scores = np.asarray(get_label_quality_scores(labels=labels, pred_probs=pred_probs))
    frac = len(issues) / len(df)
    mean_pred = float((pred_probs >= 0.5).sum(axis=1).mean())
    mean_given = float(Y.sum(axis=1).mean())
    print(f"  flagged: {len(issues)}/{len(df)} ({frac * 100:.1f}%)  |  median quality: {np.median(scores):.3f}")
    print(f"  probe coverage: predicts {mean_pred:.2f} vs {mean_given:.2f} given labels/row")
    return {"name": name, "df": df, "pred_probs": pred_probs, "issues": issues,
            "scores": scores, "frac": frac, "mean_pred": mean_pred, "mean_given": mean_given}


def show_top_issues(result, text_col, label_cols=TARGET_LABELS, top=8):
    """Print the most suspicious rows with probabilities: given-label probs vs model's top-3.

    These top-ranked rows (highest-confidence disagreements) are where Confident Learning is
    most precise — this ranked manual-review list is the primary deliverable. Showing the
    actual probabilities (not a 0.5 cut) reveals *why* each row was flagged.
    """
    df, pp = result["df"], result["pred_probs"]
    print(f"\n### Top {top} suspected label errors -- {result['name']}")
    for i in result["issues"][:top]:
        given = {label_cols[j]: round(float(pp[i, j]), 2) for j in np.where(df[label_cols].iloc[i].to_numpy())[0]}
        top3 = {label_cols[j]: round(float(pp[i, j]), 2) for j in np.argsort(-pp[i])[:3]}
        txt = str(df[text_col].iloc[i]).replace("\n", " ")[:160]
        print(f"- given={given or '(brak)'}  |  model top3={top3}")
        print(f"  {txt}")

In [3]:
results = {name: diagnose_labels(name, df, text_col) for name, (df, text_col) in frames.items()}


=== TwitterEmo: encoding 28684 rows + 5-fold OOF probe ===
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done
  flagged: 9803/28684 (34.2%)  |  median quality: 0.494
  probe coverage: predicts 0.58 vs 1.04 given labels/row

=== GoEmotions PL: encoding 34276 rows + 5-fold OOF probe ===
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done
  flagged: 9241/34276 (27.0%)  |  median quality: 0.545
  probe coverage: predicts 0.35 vs 0.79 given labels/row

=== CLARIN-Emo: encoding 6367 rows + 5-fold OOF probe ===
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done
  flagged: 1748/6367 (27.5%)  |  median quality: 0.508
  probe coverage: predicts 1.33 vs 1.71 given labels/row


In [ ]:
summary = pd.DataFrame([
    {"Dataset": r["name"], "Rows": len(r["df"]), "Flagged": len(r["issues"]),
     "Flagged %": round(r["frac"] * 100, 1),
     "Median quality": round(float(np.median(r["scores"])), 3),
     "Probe pred/row": round(r["mean_pred"], 2),
     "Given/row": round(r["mean_given"], 2)}
    for r in results.values()
])
print(summary.to_string(index=False))

print("\nNOTE — do NOT rank datasets by Flagged % / Median quality.")
print("Both are confounded by (1) label density (Given/row) and (2) how well the probe")
print("covers each dataset (Probe pred/row). The probe covers GoEmotions worst (translated,")
print("noisier Polish), so its LOW flag rate is NOT evidence of cleaner labels — the opposite")
print("of the naive 'translation = more noise' expectation, and an artifact, not a finding.")
print("Use these numbers only alongside coverage; draw conclusions from the top-K lists")
print("(qualitative) and the before/after translation-fix comparison below.")

In [5]:
for name, (df, text_col) in frames.items():
    show_top_issues(results[name], text_col=text_col)


### Top 8 suspected label errors -- TwitterEmo
- given={'zaufanie': 0.15, 'strach': 0.0, 'zdziwienie': 0.03}  |  model top3={'gniew': 0.29, 'wstręt': 0.28, 'radość': 0.26}
  @anonymized_account @anonymized_account @anonymized_account @anonymized_account Za załatwienie sprawy z tym hejterem, który Panią nękał, to mam szacunek.
- given={'smutek': 0.03, 'gniew': 0.03, 'zdziwienie': 0.01}  |  model top3={'przeczuwanie': 0.63, 'wstręt': 0.09, 'radość': 0.07}
  Jak zobaczyłem po wizycie w sklepie po ile są pomidory, to chyba go przybije do deski do krojenia, powieszę na ścianie i będę tylko wąchał 
- given={'smutek': 0.01, 'gniew': 0.06, 'zdziwienie': 0.02}  |  model top3={'radość': 0.41, 'zaufanie': 0.19, 'przeczuwanie': 0.15}
  @anonymized_account Taka Albania a we wszystkich dziedzinach lepsi od nas
- given={'zaufanie': 0.01, 'wstręt': 0.52, 'zdziwienie': 0.1}  |  model top3={'przeczuwanie': 0.96, 'wstręt': 0.52, 'gniew': 0.45}
  Czyli konfederaci się szczepią, a ordo iurki rozwodzą? Lew

---
## Effect of the translation fix (before vs after)

Same probe run on the original NLLB translations and on the fixed ones; metrics compared **only
on the rows the fix changed**. Unchanged rows act as a control.

Heavy cell: two full encodes (~43k rows) + two 5-fold OOF runs.

In [6]:
import ast

# GoEmotions label remap (kept in sync with 01_data_preparation.ipynb).
GO_EMOTIONS_LABELS = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
LABEL_MAPPING = {
    "radość":       ["joy", "amusement", "excitement", "pride", "relief", "love", "admiration", "gratitude"],
    "smutek":       ["sadness", "disappointment", "grief", "remorse", "embarrassment"],
    "zaufanie":     ["approval", "caring"],
    "wstręt":       ["disgust"],
    "strach":       ["fear", "nervousness"],
    "gniew":        ["anger", "annoyance", "disapproval"],
    "przeczuwanie": ["optimism", "desire"],
    "zdziwienie":   ["surprise", "confusion", "realization", "curiosity"],
}


def remap_go(df):
    """GoEmotions 28-id labels -> 8 Plutchik binary columns (same logic as 01)."""
    lab = df["labels"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    names = lab.apply(lambda ids: {GO_EMOTIONS_LABELS[i] for i in ids})
    Y = np.zeros((len(df), len(TARGET_LABELS)), dtype=int)
    for ci, emo in enumerate(TARGET_LABELS):
        src = set(LABEL_MAPPING[emo])
        Y[:, ci] = names.apply(lambda n: int(bool(src & n))).to_numpy()
    return Y


RAW_DIR = Path("../data/raw")
orig = pd.read_csv(RAW_DIR / "go_emotions_pl_nllb_final.csv")    # before any fix
fixed = pd.read_csv(RAW_DIR / "go_emotions_pl_nllb_fixed.csv")   # after Fix 1 + Fix 2

# Align row-for-row; keep rows with a non-empty translation in BOTH versions.
keep = (orig["text_pl"].fillna("").str.strip() != "") & (fixed["text_pl"].fillna("").str.strip() != "")
orig, fixed = orig[keep].reset_index(drop=True), fixed[keep].reset_index(drop=True)
Y = remap_go(fixed)                                    # labels are identical in both versions
changed = orig["text_pl"].to_numpy() != fixed["text_pl"].to_numpy()
print(f"Rows: {len(fixed)}  |  changed by translation fixes: {int(changed.sum())}")


def run_probe(texts):
    X = emb_model.encode(list(texts), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    pp = compute_oof_pred_probs(X, Y)
    labels = [list(np.where(r)[0]) for r in Y]
    flagged = np.zeros(len(Y), dtype=bool)
    flagged[find_label_issues(labels=labels, pred_probs=pp, return_indices_ranked_by="self_confidence")] = True
    quality = np.asarray(get_label_quality_scores(labels=labels, pred_probs=pp))
    return flagged, quality


print("\nBEFORE (original NLLB translation):")
flag_b, q_b = run_probe(orig["text_pl"].fillna(""))
print("AFTER (Fix 1 + Fix 2 translation):")
flag_a, q_a = run_probe(fixed["text_pl"].fillna(""))

ch = changed
print("\n=== Effect of the translation fixes, measured ON THE CHANGED ROWS ===")
print(f"  changed rows:           {int(ch.sum())}")
print(f"  flagged BEFORE:         {flag_b[ch].mean() * 100:5.1f}%")
print(f"  flagged AFTER:          {flag_a[ch].mean() * 100:5.1f}%")
print(f"  median quality BEFORE:  {np.median(q_b[ch]):.3f}")
print(f"  median quality AFTER:   {np.median(q_a[ch]):.3f}")
print("\nControl — unchanged rows (should barely move):")
print(f"  flagged BEFORE / AFTER: {flag_b[~ch].mean() * 100:.1f}% / {flag_a[~ch].mean() * 100:.1f}%")

Rows: 43407  |  changed by translation fixes: 5866

BEFORE (original NLLB translation):
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done
AFTER (Fix 1 + Fix 2 translation):
    fold 1/5 done
    fold 2/5 done
    fold 3/5 done
    fold 4/5 done
    fold 5/5 done

=== Effect of the translation fixes, measured ON THE CHANGED ROWS ===
  changed rows:           5866
  flagged BEFORE:          41.3%
  flagged AFTER:           27.8%
  median quality BEFORE:  0.391
  median quality AFTER:   0.479

Control — unchanged rows (should barely move):
  flagged BEFORE / AFTER: 26.2% / 26.3%


## How to read this

- **Flagged %** is an upper bound and is confounded across datasets by label density and probe
  coverage — it is **not** a clean cross-dataset noise ranking.
- **Probe pred/row vs Given/row** = coverage; the closer, the more trustworthy the flags.
- **Median quality** = per-row cleanlab score (1 = clean). Same confounding caveat.
- The ranked top-K list is the primary output.

---
## Is the diagnostic actionable? Pruning vs random (TwitterEmo)

Drop the worst X% of **train** rows per cleanlab, fit, evaluate on the untouched test set.
**Control:** drop the same number of *random* train rows — the diagnostic only adds value if it
beats random. Quality scores taken on train only; thresholds tuned on val per variant.

In [ ]:
from sklearn.metrics import f1_score

# Maps the in-memory dataset name -> (file prefix, raw text column) for val/test.
VAL_TEST_TEXT = {
    "TwitterEmo":    ("twitteremo",  "tekst"),
    "GoEmotions PL": ("go_emotions", "text_pl"),
    "CLARIN-Emo":    ("clarin_emo",  "tekst"),
}


def _Y(df):
    return df[TARGET_LABELS].to_numpy().astype(int)


def _tune_thr(yv, pv):
    # thesis_lib.optimal_thresholds; 04 używa rzadszej siatki step=0.05 (zachowane jawnie).
    return optimal_thresholds(yv, pv, step=0.05)


_f1_macro_ci = bootstrap_f1_ci  # thesis_lib (identyczna implementacja bootstrap 95% CI F1-Macro)


def prune_experiment(name, fracs=(0.05, 0.10, 0.20)):
    """Does removing the lowest-quality TRAIN rows (cleanlab) beat removing random rows?

    Trains an embedding+LogReg probe on the pruned train, evaluates on the untouched test.
    Random pruning of the same size is the control: cleanlab only adds value if it beats it.
    """
    prefix, tcol = VAL_TEST_TEXT[name]
    df_tr, _ = frames[name]
    scores = results[name]["scores"]                       # per-train-row quality (computed above)
    df_va = pd.read_csv(PROCESSED_DIR / f"{prefix}_val.csv")
    df_te = pd.read_csv(PROCESSED_DIR / f"{prefix}_test.csv")
    print(f"### Pruning experiment: {name}  (train={len(df_tr)}, test={len(df_te)})")

    enc = lambda d: emb_model.encode(d[tcol].fillna("").tolist(), batch_size=64,
                                     show_progress_bar=False, normalize_embeddings=True)
    Xtr, Xva, Xte = enc(df_tr), enc(df_va), enc(df_te)
    ytr, yva, yte = _Y(df_tr), _Y(df_va), _Y(df_te)
    n = len(ytr)
    worst = np.argsort(scores)                             # ascending: lowest quality first
    rng = np.random.default_rng(RANDOM_STATE)

    def fit_eval(keep):
        clf = OneVsRestClassifier(LogisticRegression(max_iter=1000)).fit(Xtr[keep], ytr[keep])
        thr = _tune_thr(yva, clf.predict_proba(Xva))       # thresholds on val only
        return (clf.predict_proba(Xte) >= thr).astype(int)

    b0, l0, h0 = _f1_macro_ci(yte, fit_eval(np.arange(n)))
    rows = [{"prune%": 0, "n_train": n, "strategy": "baseline",
             "F1_macro": round(b0, 3), "95% CI": f"[{l0:.3f}, {h0:.3f}]"}]
    for fr in fracs:
        k = int(fr * n)
        keep_q = np.setdiff1d(np.arange(n), worst[:k])
        bq, lq, hq = _f1_macro_ci(yte, fit_eval(keep_q))
        rows.append({"prune%": int(fr * 100), "n_train": len(keep_q), "strategy": "cleanlab",
                     "F1_macro": round(bq, 3), "95% CI": f"[{lq:.3f}, {hq:.3f}]"})
        keep_r = np.setdiff1d(np.arange(n), rng.choice(n, k, replace=False))
        br, lr, hr = _f1_macro_ci(yte, fit_eval(keep_r))
        rows.append({"prune%": int(fr * 100), "n_train": len(keep_r), "strategy": "random",
                     "F1_macro": round(br, 3), "95% CI": f"[{lr:.3f}, {hr:.3f}]"})
    return pd.DataFrame(rows)


# Primary run: TwitterEmo (native, crowd-labeled -> real annotation noise to clean).
prune_tw = prune_experiment("TwitterEmo")
print(prune_tw.to_string(index=False))

# Optional contrasts (uncomment to run):
# print(prune_experiment("CLARIN-Emo").to_string(index=False))   # expert -> negative control
# print(prune_experiment("GoEmotions PL").to_string(index=False))

---
## Podsumowanie

1. **Ranking szumu między zbiorami** — nieużyteczny, skonfundowany gęstością etykiet i pokryciem sondy.
2. **Walidacja naprawy tłumaczeń** — naprawa obniżyła szum: na zmienionych wierszach flagi 41,3% → 27,8%,
   mediana jakości 0,391 → 0,479, przy płaskiej grupie kontrolnej (26,2% → 26,3%).
3. **Czy czyszczenie poprawia model** — nie. Przycinanie wg cleanlab było gorsze od losowego
   (10%: 0,431 vs 0,460), a każde przycinanie ≤ baseline (0,461).

Confident Learning ma tu wartość wyłącznie jakościową. `data/processed/` pozostaje nietknięte.